# 37-D. 대화코퍼스에서 모음조화/모음충돌회피 검색

37번 사전검색 결과의 용언이 실제 대화에서 어떻게 활용되는지 확인

## 목표
- 37번에서 추출한 모음충돌회피 후보(모음어간 용언)가 대화에서 실제로 나타나는지 검색
- form vs pronunciation 비교: 실제 발음에서 활음화/탈락/활음삽입 중 어떤 전략이 실현되는지
- 화자 변수(성별, 연령, 출생지)별 변이 분석

## 입력
- 37번 결과: `search_results/vowel_harmony_collision_all_*.csv`
- 대화 enriched 샘플: `01_nikl_dialogue_enriched_sample3k.csv` (3.1MB)
- 대화 enriched 전체: `01_nikl_dialogue_enriched.csv` (4.7GB)

## 출력
- 대화 매칭 CSV: `search_results/dialogue_vowel_collision_*.csv`

## 1. 환경 설정

In [1]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DATA_2026'
except ImportError:
    PROJECT_ROOT = os.path.dirname(os.getcwd())
    print(f'Local mode: {PROJECT_ROOT}')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re

# 경로 설정
SEARCH_37 = f'{PROJECT_ROOT}/37_vowel_harmony_collision/search_results'
DIALOGUE_SAMPLE = f'{PROJECT_ROOT}/00_raw_data/04_nikl_dialogue/02_csv/01_nikl_dialogue_enriched_sample3k.csv'
DIALOGUE_FULL = f'{PROJECT_ROOT}/00_raw_data/04_nikl_dialogue/02_csv/01_nikl_dialogue_enriched.csv'
RESULT_DIR = f'{PROJECT_ROOT}/37_vowel_harmony_collision/search_results'
os.makedirs(RESULT_DIR, exist_ok=True)

# 샘플 vs 전체 선택
USE_SAMPLE = True  # True: 3K 샘플, False: 4.7GB 전체
DIALOGUE_PATH = DIALOGUE_SAMPLE if USE_SAMPLE else DIALOGUE_FULL
print(f'데이터: {"sample 3K" if USE_SAMPLE else "full 460만"}')
print(f'{DIALOGUE_PATH}')

데이터: sample 3K
/content/drive/MyDrive/DATA_2026/00_raw_data/04_nikl_dialogue/02_csv/01_nikl_dialogue_enriched_sample3k.csv


## 2. 37번 결과 로드 — 어간 리스트

In [3]:
# 37번 결과 로드
all_files = sorted(Path(SEARCH_37).glob('vowel_harmony_collision_all_*.csv'))
if not all_files:
    raise FileNotFoundError('37번 결과 CSV가 없습니다')

latest = all_files[-1]
print(f'사용 파일: {latest.name}')

df37 = pd.read_csv(latest, encoding='utf-8-sig')
print(f'37번 결과: {len(df37):,}행')
print(f'environment: {df37["environment"].value_counts().to_dict()}')

# 어간 → 정보 매핑 딕셔너리
stem_info = {}
for _, row in df37.iterrows():
    stem = row['stem']
    if pd.isna(stem):
        continue
    stem_info[stem] = {
        'word': row['word'],
        'environment': row['environment'],
        'collision_type': row.get('collision_type', ''),
        'stem_final_vowel': row.get('stem_final_vowel_name', ''),
        'vowel_polarity': row.get('vowel_polarity', ''),
    }

stems = list(stem_info.keys())
stem_set = set(stems)
print(f'고유 어간: {len(stems):,}개')

사용 파일: vowel_harmony_collision_all_20260312_071836.csv
37번 결과: 93,413행
environment: {'vowel_collision': 87271, 'vowel_harmony': 6142}
고유 어간: 59,597개


## 3. 대화 코퍼스 로드

In [4]:
print(f'대화 코퍼스 로딩... ({"sample" if USE_SAMPLE else "full"})')
df_dial = pd.read_csv(DIALOGUE_PATH, encoding='utf-8-sig', low_memory=False)
print(f'대화 코퍼스: {len(df_dial):,}행')
print(f'컬럼: {list(df_dial.columns)[:15]}...')
df_dial.head(3)

대화 코퍼스 로딩... (sample)
대화 코퍼스: 2,999행
컬럼: ['file_id', 'year', 'category', 'doc_id', 'doc_title', 'doc_date', 'topic', 'speaker_id', 'speaker_age', 'speaker_sex', 'speaker_occupation', 'speaker_birthplace', 'speaker_principal_residence', 'speaker_current_residence', 'speaker_education']...


,file_id,year,category,doc_id,doc_title,doc_date,topic,speaker_id,speaker_age,speaker_sex,...,original_form,start,end,note,pronunciation,form_roman,morphs,morphs_roman,morphs_v7_ids,morphs_v7_origins
0,SDRW2000000004,2020,구어 > 사적 대화 > 일상대화,SDRW2000000004.1,2인 일상 대화,20200602,"반려동물 > 영상, 파충류, 길고양이, 수술",SD2000006,20대,여성,...,이용이 되는 느낌을 좀 많이 받았어서,445.57208,448.75106,NaN,I iO NG I D E N EU N N EU GG I M EU L J O N M ...,I iO NG I D oE N EU N N EU GG I M EU L J O M M...,이용/NNG+이/JKS 되/VV+는/ETM 느낌/NNG+을/JKO 좀/MAG 많이/...,I iO NG/NNG+I/JKS D oE/VV+N EU N/ETM N EU GG I...,"이용:746158:001,746158:002,746159:003,746160:004...",利用|||||
1,SDRW2000000008,2020,구어 > 사적 대화 > 일상대화,SDRW2000000008.1,2인 일상 대화,20200603,"계절/날씨 > 사계절, 좋아하는 계절, 태풍 피해",SD2000014,20대,여성,...,돌아다니기만 해도 꽃들이 자꾸,147.37901,150.62301,NaN,D O R A D A N E G I M A uEO N H E D O GG O t D...,D O L A D A N I G I M A N H E D O GG O t D EU ...,돌아다니/VV+기/ETN+만/JX 하/VX+아도/EC 꽃/NNG+들/XSN+이/JK...,D O L A D A N I/VV+G I/ETN+M A N/JX H A/VX+A D...,"돌아다니:243769:001,243769:002|하:1100497:034,11004...",|||
2,SDRW2000000011,2020,구어 > 사적 대화 > 일상대화,SDRW2000000011.1,2인 일상 대화,20200603,"여행지(국내/해외) > 장소, 첫여행, 추천, 동행자, 돈, 인종차별",SD2000020,30대,여성,...,인도라는 나란 어떤가요?,885.45601,887.37206,NaN,I N D O O R A N EU N N EO R EO N O t DD O NG G...,I N D O R A N EU N N A R A N EO DD EO N G A iO,인도/NNP+이/VCP+라는/ETM 나란/NNG 어떻/VA+ㄴ가/EF+요/JX+?/SF,I N D O/NNP+I/VCP+R A N EU N/ETM N A R A N/NNG...,어떻:632856:001,NaN


## 4. 어간 매칭 — form에서 용언 활용형 검색

대화 코퍼스의 form(어절) 또는 형태소 분석에서 37번 어간을 포함하는 발화 검색

In [5]:
def extract_verb_stem_from_morphs(morphs_str):
    """대화 코퍼스 형태소 분석에서 용언 어간 추출"""
    if pd.isna(morphs_str):
        return None
    parts = str(morphs_str).split('+')
    for part in parts:
        if '/' in part:
            morph, pos = part.rsplit('/', 1)
            if pos in ('VV', 'VA', 'VX'):
                return morph
    return None

def extract_suffix_from_morphs(morphs_str):
    """형태소 분석에서 용언 다음 어미 추출"""
    if pd.isna(morphs_str):
        return None
    parts = str(morphs_str).split('+')
    found_verb = False
    for part in parts:
        if '/' in part:
            morph, pos = part.rsplit('/', 1)
            if found_verb and pos in ('EC', 'EF', 'EP', 'ETN', 'ETM'):
                return morph
            if pos in ('VV', 'VA', 'VX'):
                found_verb = True
    return None

# 형태소 분석 컬럼 확인
morph_col = None
for candidate in ['morpheme_analysis', 'morphs', 'morph_analysis']:
    if candidate in df_dial.columns:
        morph_col = candidate
        break

if morph_col:
    print(f'형태소 분석 컬럼: {morph_col}')
    df_dial['verb_stem'] = df_dial[morph_col].apply(extract_verb_stem_from_morphs)
    df_dial['verb_suffix'] = df_dial[morph_col].apply(extract_suffix_from_morphs)
    n_verb = df_dial['verb_stem'].notna().sum()
    print(f'용언 포함 행: {n_verb:,} / {len(df_dial):,}')
else:
    # 형태소 분석 없으면 form에서 직접 검색
    print('형태소 분석 컬럼 없음 → form 기반 검색')
    df_dial['verb_stem'] = None
    df_dial['verb_suffix'] = None

형태소 분석 컬럼: morphs
용언 포함 행: 2,204 / 2,999


In [6]:
# 37번 어간 매칭
if morph_col:
    # 형태소 기반 매칭
    df_matched = df_dial[df_dial['verb_stem'].isin(stem_set)].copy()
else:
    # form 기반 매칭 (어간이 form에 포함되는 경우)
    # 빈도 높은 어간 우선 (속도 위해 제한)
    freq_stems = df37.nlargest(500, 'freq_LS_total')['stem'].unique().tolist()
    mask = df_dial['form'].apply(
        lambda x: any(s in str(x) for s in freq_stems) if pd.notna(x) else False
    )
    df_matched = df_dial[mask].copy()

print(f'37번 어간 매칭: {len(df_matched):,}행')
if 'verb_stem' in df_matched.columns:
    print(f'매칭된 고유 어간: {df_matched["verb_stem"].nunique():,}개 / {len(stem_set):,}개')

37번 어간 매칭: 363행
매칭된 고유 어간: 178개 / 59,597개


In [ ]:
# 37번 정보 병합
if 'verb_stem' in df_matched.columns:
    df_matched['dict_word'] = df_matched['verb_stem'].map(lambda s: stem_info.get(s, {}).get('word', ''))
    df_matched['dict_environment'] = df_matched['verb_stem'].map(lambda s: stem_info.get(s, {}).get('environment', ''))
    df_matched['dict_collision_type'] = df_matched['verb_stem'].map(lambda s: stem_info.get(s, {}).get('collision_type', ''))
    df_matched['dict_stem_final_vowel'] = df_matched['verb_stem'].map(lambda s: stem_info.get(s, {}).get('stem_final_vowel', ''))
    df_matched['dict_vowel_polarity'] = df_matched['verb_stem'].map(lambda s: stem_info.get(s, {}).get('vowel_polarity', ''))

# include 컬럼 추가 (수동 체크용)
df_matched['include'] = ''

print('37번 정보 병합 완료')
cols_show = ['utterance_id', 'form', 'verb_stem', 'verb_suffix', 'dict_word', 'dict_environment']
cols_show = [c for c in cols_show if c in df_matched.columns]
print(df_matched[cols_show].head(10).to_string())

## 5. 모음충돌 어미 필터 + form vs pronunciation 비교

In [8]:
# 모음충돌 관련 어미 필터
collision_suffixes = {'아', '어', '아서', '어서', '아도', '어도', '아야', '어야',
                      '았', '었', '아요', '어요', '아라', '어라', '아지', '어지'}

if 'verb_suffix' in df_matched.columns:
    df_collision = df_matched[df_matched['verb_suffix'].isin(collision_suffixes)].copy()
    print(f'모음충돌 어미 결합: {len(df_collision):,}행 / {len(df_matched):,}행')
    print(f'\n어미 분포:')
    print(df_collision['verb_suffix'].value_counts().head(10))
else:
    df_collision = df_matched.copy()
    print(f'어미 필터 불가 → 전체 사용: {len(df_collision):,}행')

모음충돌 어미 결합: 102행 / 363행

어미 분포:
verb_suffix
아     28
었     27
았     20
아서    10
어      8
아요     4
어서     2
어요     2
어도     1
Name: count, dtype: int64


In [9]:
# form vs pronunciation 비교
pron_col = None
for candidate in ['pronunciation', 'pron', 'prono']:
    if candidate in df_collision.columns:
        pron_col = candidate
        break

if pron_col:
    df_collision['form_pron_diff'] = df_collision['form'] != df_collision[pron_col]
    n_diff = df_collision['form_pron_diff'].sum()
    print(f'철자!=발음: {n_diff:,}행 ({n_diff/max(len(df_collision),1)*100:.1f}%)')

    # 차이 예시
    diff_ex = df_collision[df_collision['form_pron_diff']][['form', pron_col, 'verb_stem', 'verb_suffix']].head(20)
    print(f'\n차이 예시:')
    print(diff_ex.to_string())
else:
    print('발음 컬럼 없음')

철자!=발음: 102행 (100.0%)

차이 예시:
                                    form                                                                                                               pronunciation verb_stem verb_suffix
11                                 사드렸는데                                                                                                 S A D EU R iEO N N EU N D E         사           었
36                                   싸워서                                                                                                           EU S A uEO S O iO        싸우          어서
100                               그렇게 해서                                                                                                  G EU R EO Kh E H E S EO iO        그렇          아서
187             배워야 될 부분에 대해서는 저희는 이런 거는                                          G oE oA D oE L B U B U N E D E H E S EO N EU N S EO I N EU N I R EO N G uEO N EU N        배우          아서
248                          선물해드리고

## 6. 화자 변수별 변이 분석

In [10]:
# 화자 변수별 통계
for col in ['speaker_sex', 'speaker_age', 'speaker_birthplace']:
    if col in df_collision.columns:
        print(f'\n=== {col} ===')
        print(df_collision[col].value_counts().head(10))

        if pron_col and 'form_pron_diff' in df_collision.columns:
            var_stats = df_collision.groupby(col)['form_pron_diff'].agg(['sum', 'count', 'mean'])
            var_stats.columns = ['변이_건수', '전체_건수', '변이율']
            print(var_stats.sort_values('전체_건수', ascending=False).head(10))


=== speaker_sex ===
speaker_sex
여성    72
남성    30
Name: count, dtype: int64
             변이_건수  전체_건수  변이율
speaker_sex                   
여성              72     72  1.0
남성              30     30  1.0

=== speaker_age ===
speaker_age
20대       36
30대       20
10대       15
40대       14
50대       13
60대 이상     4
Name: count, dtype: int64
             변이_건수  전체_건수  변이율
speaker_age                   
20대             36     36  1.0
30대             20     20  1.0
10대             15     15  1.0
40대             14     14  1.0
50대             13     13  1.0
60대 이상           4      4  1.0

=== speaker_birthplace ===
speaker_birthplace
서울    25
부산    13
경기    13
대전     8
대구     6
충남     5
울산     4
경남     4
인천     4
경북     4
Name: count, dtype: int64
                    변이_건수  전체_건수  변이율
speaker_birthplace                   
서울                     25     25  1.0
경기                     13     13  1.0
부산                     13     13  1.0
대전                      8      8  1.0
대구                     

## 7. 환경별 통계

In [11]:
if 'dict_environment' in df_collision.columns:
    print('=== environment별 대화 출현 ===')
    print(df_collision['dict_environment'].value_counts())

if 'dict_collision_type' in df_collision.columns:
    print('\n=== collision_type별 대화 출현 ===')
    print(df_collision['dict_collision_type'].value_counts())

if 'dict_stem_final_vowel' in df_collision.columns:
    print('\n=== 어간말모음별 대화 출현 ===')
    print(df_collision['dict_stem_final_vowel'].value_counts())

=== environment별 대화 출현 ===
dict_environment
vowel_collision    58
vowel_harmony      44
Name: count, dtype: int64

=== collision_type별 대화 출현 ===
dict_collision_type
none               44
glide_insertion    35
glide_formation    12
deletion           11
Name: count, dtype: int64

=== 어간말모음별 대화 출현 ===
dict_stem_final_vowel
ㅏ    47
ㅓ    25
ㅣ    10
ㅡ     7
ㅗ     6
ㅜ     4
ㅢ     1
ㅚ     1
ㅟ     1
Name: count, dtype: int64


## 8. 결과 저장

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
suffix = 'sample' if USE_SAMPLE else 'full'

# 전체 매칭 저장
if len(df_matched) > 0:
    out1 = f'{RESULT_DIR}/dialogue_vowel_collision_all_{suffix}_{timestamp}.csv'
    df_matched.to_csv(out1, index=False, encoding='utf-8-sig')
    print(f'전체: {out1} ({len(df_matched):,}행)')

# 모음충돌 어미만 저장
if len(df_collision) > 0:
    out2 = f'{RESULT_DIR}/dialogue_vowel_collision_suffix_{suffix}_{timestamp}.csv'
    df_collision.to_csv(out2, index=False, encoding='utf-8-sig')
    print(f'충돌어미: {out2} ({len(df_collision):,}행)')

print('\n저장 완료')